<a href="https://colab.research.google.com/github/arulbenjaminchandru/ai-architect-program/blob/main/Day_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PDF Q&A Lab: RAG with ChromaDB + Claude Haiku

**Goal:** Upload a PDF → chunk it → embed with MPNet → store in ChromaDB →
ask a question → retrieve relevant chunks → get a grounded answer from Claude Haiku.

We will build this step by step, then explore three different retrieval strategies:
1. **Dense Retrieval** (vector similarity — the baseline)
2. **BM25 Sparse Retrieval** (keyword matching)
3. **Hybrid Retrieval** (Dense + BM25 fused together)

---

> **Setup:** Add your Anthropic API key to Colab Secrets (🔑 icon on the left sidebar)  
> with the name `ANTHROPIC_API_KEY` before running any cells.


## Step 0 — Install Libraries

In [ ]:
!pip uninstall -y \
  google-adk \
  opentelemetry-exporter-gcp-logging \
  opentelemetry-exporter-otlp-proto-http

In [ ]:
# Install everything needed for this lab
# Run this cell first; Colab will ask to restart the runtime — click Restart

!pip install -q chromadb sentence-transformers pdfplumber anthropic rank-bm25
print("✅ All libraries installed")


## Step 1 — Load Your Anthropic API Key

In [ ]:
# Read the API key from Colab Secrets
# Make sure you added ANTHROPIC_API_KEY in the 🔑 Secrets panel

from google.colab import userdata  # Colab's secret store
import os                          # To set environment variables

# Pull the key from Colab Secrets and expose it as an environment variable
# (the Anthropic SDK automatically reads ANTHROPIC_API_KEY from the environment)
os.environ["ANTHROPIC_API_KEY"] = userdata.get("MY_API_KEY")

print("✅ API key loaded")


## Step 2 — Upload Your PDF

Run this cell. A file picker will appear — choose any PDF from your computer.
The PDF path is saved to `PDF_PATH` for use in later steps.


In [ ]:
from google.colab import files  # Colab's file upload helper

print("A file picker will open below — select your PDF.")
uploaded = files.upload()  # Opens the upload dialog; returns {filename: bytes}

# Grab the filename of the uploaded file
PDF_PATH = list(uploaded.keys())[0]  # First (and only) uploaded filename

print(f"\n✅ Uploaded: {PDF_PATH}")
print(f"   File size: {len(uploaded[PDF_PATH]) / 1024:.1f} KB")


## Step 3 — Extract Text from the PDF

`pdfplumber` reads each page and extracts its text content.
After extraction we do basic cleaning: collapse extra spaces,
fix hyphenated line breaks, and normalise blank lines.


In [ ]:
import pdfplumber  # PDF text extraction library
import re          # Regular expressions for text cleaning

def extract_and_clean(pdf_path):
    """
    Open a PDF, extract text from every page, and return cleaned text.

    Args:
        pdf_path (str): Path to the PDF file

    Returns:
        str: Cleaned full-document text
    """
    raw_pages = []  # Collect each page's raw text

    with pdfplumber.open(pdf_path) as pdf:  # Open the PDF
        total_pages = len(pdf.pages)         # How many pages does it have?
        print(f"PDF has {total_pages} pages")

        for i, page in enumerate(pdf.pages):       # Iterate over pages
            text = page.extract_text()             # Extract text from this page
            if text:                               # Some pages (images) return None
                raw_pages.append(text)             # Collect non-empty pages
                print(f"  Page {i + 1}: {len(text)} chars")
            else:
                print(f"  Page {i + 1}: (no extractable text — possibly scanned)")

    # Join all pages with a double newline separator
    full_text = "\n\n".join(raw_pages)

    # ── Cleaning ──
    full_text = re.sub(r" +", " ", full_text)         # Collapse multiple spaces → one
    full_text = re.sub(r"-\n", "", full_text)          # Fix "hyphen-\nbreak" → "hyphenbreak"
    full_text = re.sub(r"\n{3,}", "\n\n", full_text)  # Collapse 3+ newlines → 2
    full_text = full_text.strip()                      # Remove leading/trailing whitespace

    return full_text  # Return the cleaned document text


# Run extraction on the uploaded PDF
DOCUMENT_TEXT = extract_and_clean(PDF_PATH)

print(f"\n✅ Extraction complete")
print(f"   Total characters : {len(DOCUMENT_TEXT)}")
print(f"   Estimated tokens : {len(DOCUMENT_TEXT) // 4}")
print(f"\nFirst 500 characters of extracted text:")
print("-" * 60)
print(DOCUMENT_TEXT[:500])
print("-" * 60)


## Step 4 — Chunk the Document

We split the document into overlapping fixed-size windows.

- **Chunk size:** 400 tokens ≈ 1 600 characters (sweet spot for most PDFs)
- **Overlap:** 50 tokens ≈ 200 characters (prevents losing context at boundaries)

Each chunk will get its own embedding so retrieval can zoom in on the right section.


In [ ]:
def chunk_text(text, chunk_size_tokens=400, overlap_tokens=50):
    """
    Split text into overlapping fixed-size windows.

    Args:
        text (str): Full document text
        chunk_size_tokens (int): Target chunk size in tokens (1 token ≈ 4 chars)
        overlap_tokens (int): Overlap between consecutive chunks in tokens

    Returns:
        list[str]: List of text chunks
    """
    char_size    = chunk_size_tokens * 4   # Convert tokens → characters
    char_overlap = overlap_tokens    * 4   # Convert overlap tokens → characters

    chunks = []  # Accumulate chunks here
    start  = 0   # Sliding window start position

    while start < len(text):                           # Slide through the document
        end   = min(start + char_size, len(text))      # Window end (capped at EOF)
        chunk = text[start:end]                        # Slice out this window
        if chunk.strip():                              # Skip empty/whitespace-only chunks
            chunks.append(chunk.strip())               # Store the chunk
        start = start + char_size - char_overlap       # Advance window (with overlap)

    return chunks  # Return all chunks


# Chunk the document
CHUNKS = chunk_text(DOCUMENT_TEXT, chunk_size_tokens=400, overlap_tokens=50)

print(f"✅ Chunking complete")
print(f"   Number of chunks      : {len(CHUNKS)}")
print(f"   Avg chunk length      : {sum(len(c) for c in CHUNKS) // len(CHUNKS)} chars")
print(f"   ≈ Avg chunk tokens    : {sum(len(c) for c in CHUNKS) // len(CHUNKS) // 4}")
print(f"\nFirst chunk preview:")
print("-" * 60)
print(CHUNKS[0][:400])
print("-" * 60)


## Step 5 — Embed Chunks with MPNet

`all-mpnet-base-v2` maps each chunk to a 768-dimensional vector.
Chunks that are semantically similar will have vectors that point
in similar directions — that's what lets us retrieve by meaning.


In [ ]:
from sentence_transformers import SentenceTransformer  # Embedding library
import numpy as np                                      # Numerical arrays

# Load the MPNet model (downloads ~438 MB on first run; cached after that)
print("Loading MPNet embedding model (may download on first run)...")
EMBED_MODEL = SentenceTransformer("all-mpnet-base-v2")
print("✅ Model loaded — embedding dimension:", EMBED_MODEL.get_sentence_embedding_dimension())

# Embed all chunks in a single batch call (faster than one at a time)
print(f"\nEmbedding {len(CHUNKS)} chunks...")
CHUNK_EMBEDDINGS = EMBED_MODEL.encode(
    CHUNKS,                # List of strings to embed
    show_progress_bar=True # Display a progress bar
)

print(f"\n✅ Embeddings ready")
print(f"   Shape: {CHUNK_EMBEDDINGS.shape}")  # Should be (num_chunks, 768)
print(f"   Each chunk → a vector of {CHUNK_EMBEDDINGS.shape[1]} numbers")


## Step 6 — Store in ChromaDB

ChromaDB is an in-memory vector database.
We insert each chunk's text, its embedding, and metadata
(chunk index + approximate token count) so we can retrieve
and filter later.


In [ ]:
import chromadb  # Vector database

# Create an in-memory ChromaDB client (data lives in RAM; no files written)
CHROMA_CLIENT = chromadb.Client()

# Create a collection — think of it as a table for vector data
COLLECTION = CHROMA_CLIENT.create_collection(
    name="pdf_chunks",                              # Name for this collection
    metadata={"description": "Chunks from the uploaded PDF"}
)

# Build per-chunk IDs and metadata
chunk_ids = [f"chunk_{i}" for i in range(len(CHUNKS))]  # Unique string ID per chunk

chunk_metadatas = [
    {
        "chunk_index"       : i,                    # Position of this chunk in the document
        "approx_tokens"     : len(CHUNKS[i]) // 4,  # Rough token count
        "source"            : PDF_PATH              # Which file this came from
    }
    for i in range(len(CHUNKS))
]

# Insert everything into ChromaDB in one call
COLLECTION.add(
    ids        = chunk_ids,                         # String IDs (must be unique)
    documents  = CHUNKS,                            # Raw chunk text
    embeddings = CHUNK_EMBEDDINGS.tolist(),          # Vectors as Python lists
    metadatas  = chunk_metadatas                    # Metadata dicts
)

print(f"✅ Stored {COLLECTION.count()} chunks in ChromaDB")
print(f"   Each record contains: text + 768-dim embedding + metadata")


## Step 7 — Retrieval Method 1: Dense Retrieval (Vector Similarity)

The query is embedded with the same MPNet model.
ChromaDB computes cosine distance between the query vector and
every stored chunk vector, then returns the closest matches.

**Strength:** Understands meaning — "automobile" matches "car".  
**Weakness:** Misses exact keyword matches when phrasing differs greatly.


In [ ]:
def retrieve_dense(query, k=3):
    """
    Retrieve the top-k chunks most semantically similar to the query.

    Steps:
      1. Embed the query with MPNet
      2. Ask ChromaDB for the k nearest vectors (cosine distance)
      3. Return chunks, similarity scores, and metadata

    Args:
        query (str): User's question
        k (int): Number of chunks to retrieve

    Returns:
        list[dict]: Each dict has 'text', 'score', 'chunk_index'
    """
    # Embed the query — must use the same model as the chunks
    query_vec = EMBED_MODEL.encode([query])  # Returns shape (1, 768)

    # Query ChromaDB: returns k nearest neighbours by L2 / cosine distance
    results = COLLECTION.query(
        query_embeddings = query_vec.tolist(),  # The query vector
        n_results        = k                    # How many results to return
    )

    # Unpack the nested result structure (ChromaDB wraps results in an outer list)
    docs      = results["documents"][0]   # List of chunk texts
    distances = results["distances"][0]   # L2 distances (lower = more similar)
    metas     = results["metadatas"][0]   # List of metadata dicts

    # Convert L2 distance to a 0–1 similarity score (higher = more relevant)
    retrieved = []
    for doc, dist, meta in zip(docs, distances, metas):
        similarity = 1 / (1 + dist)  # Monotonically decreasing transform
        retrieved.append({
            "text"        : doc,
            "score"       : round(similarity, 4),
            "chunk_index" : meta["chunk_index"]
        })

    return retrieved  # Sorted best → worst by ChromaDB


# ── Quick test ──
print("=" * 65)
print("DENSE RETRIEVAL TEST")
print("=" * 65)
test_query = "What is the main topic of this document?"

results_dense = retrieve_dense(test_query, k=3)

for rank, r in enumerate(results_dense, start=1):
    print(f"\nRank {rank}  |  Chunk #{r['chunk_index']}  |  Score: {r['score']}")
    print(r["text"][:200] + "...")
print("=" * 65)


## Step 8 — Retrieval Method 2: BM25 Sparse Retrieval

BM25 is a classic keyword-ranking algorithm (the engine behind Elasticsearch).
It scores each chunk based on how often query terms appear, weighted by how
rare those terms are across the whole document.

**Strength:** Exact keyword match — perfect for product codes, names, specific terms.  
**Weakness:** No semantic understanding — "automobile" won't match "car".


In [ ]:
from rank_bm25 import BM25Okapi  # BM25 implementation

# Tokenise every chunk (lowercase, split on whitespace)
# BM25 works on token lists, not raw strings
tokenised_chunks = [chunk.lower().split() for chunk in CHUNKS]

# Build the BM25 index over all tokenised chunks
BM25_INDEX = BM25Okapi(tokenised_chunks)

print(f"✅ BM25 index built over {len(CHUNKS)} chunks")


def retrieve_bm25(query, k=3):
    """
    Retrieve the top-k chunks by BM25 keyword score.

    Steps:
      1. Tokenise the query
      2. Score every chunk with BM25
      3. Return top-k by score

    Args:
        query (str): User's question
        k (int): Number of chunks to retrieve

    Returns:
        list[dict]: Each dict has 'text', 'score', 'chunk_index'
    """
    query_tokens = query.lower().split()       # Tokenise query the same way as chunks
    scores       = BM25_INDEX.get_scores(query_tokens)  # Score all chunks

    # Build (score, index) pairs and sort descending
    scored_indices = sorted(
        range(len(scores)),         # Indices 0 … N-1
        key    = lambda i: scores[i],  # Sort key: BM25 score
        reverse= True               # Highest score first
    )

    top_k_indices = scored_indices[:k]  # Keep only the top-k

    retrieved = []
    for idx in top_k_indices:
        retrieved.append({
            "text"        : CHUNKS[idx],
            "score"       : round(float(scores[idx]), 4),
            "chunk_index" : idx
        })

    return retrieved  # Return top-k results


# ── Quick test ──
print("=" * 65)
print("BM25 RETRIEVAL TEST")
print("=" * 65)

results_bm25 = retrieve_bm25(test_query, k=3)

for rank, r in enumerate(results_bm25, start=1):
    print(f"\nRank {rank}  |  Chunk #{r['chunk_index']}  |  Score: {r['score']}")
    print(r["text"][:200] + "...")
print("=" * 65)


## Step 9 — Retrieval Method 3: Hybrid Retrieval (Dense + BM25)

Hybrid retrieval combines both signals using **Reciprocal Rank Fusion (RRF)**.

Each chunk receives an RRF score from both retrievers based on its rank position:

```
RRF score = 1 / (60 + rank_from_dense) + 1 / (60 + rank_from_bm25)
```

The constant 60 dampens the impact of rank differences.
Chunks that rank well in *both* systems get the highest combined scores.

**Strength:** Gets the best of both — semantic + keyword.  
**When to use:** Default choice for most production RAG systems.


In [ ]:
def retrieve_hybrid(query, k=3, rrf_k=60):
    """
    Hybrid retrieval using Reciprocal Rank Fusion of Dense + BM25 results.

    Args:
        query (str): User's question
        k (int): Number of final results to return
        rrf_k (int): RRF damping constant (default 60 is standard)

    Returns:
        list[dict]: Each dict has 'text', 'score', 'chunk_index'
    """
    # ── Step A: Get many candidates from each retriever ──
    # Retrieve more than k so the fusion has a larger pool to work with
    pool_size = min(k * 5, len(CHUNKS))  # Don't ask for more than we have

    dense_results = retrieve_dense(query, k=pool_size)   # Dense candidates
    bm25_results  = retrieve_bm25(query,  k=pool_size)   # BM25 candidates

    # ── Step B: Build rank maps (chunk_index → rank position, 0-based) ──
    dense_ranks = {r["chunk_index"]: rank for rank, r in enumerate(dense_results)}
    bm25_ranks  = {r["chunk_index"]: rank for rank, r in enumerate(bm25_results)}

    # ── Step C: Collect all unique chunk indices seen in either retriever ──
    all_indices = set(dense_ranks.keys()) | set(bm25_ranks.keys())

    # ── Step D: Compute RRF score for each unique chunk ──
    rrf_scores = {}  # chunk_index → combined RRF score

    for idx in all_indices:
        score = 0.0  # Start from zero

        if idx in dense_ranks:
            # Add contribution from dense rank
            score = score + 1.0 / (rrf_k + dense_ranks[idx] + 1)

        if idx in bm25_ranks:
            # Add contribution from BM25 rank
            score = score + 1.0 / (rrf_k + bm25_ranks[idx]  + 1)

        rrf_scores[idx] = score  # Store combined score

    # ── Step E: Sort by RRF score descending and take top-k ──
    sorted_indices = sorted(
        rrf_scores.keys(),
        key    = lambda i: rrf_scores[i],
        reverse= True
    )
    top_k_indices = sorted_indices[:k]

    retrieved = []
    for idx in top_k_indices:
        retrieved.append({
            "text"        : CHUNKS[idx],
            "score"       : round(rrf_scores[idx], 6),
            "chunk_index" : idx
        })

    return retrieved  # Return fused top-k results


# ── Quick test ──
print("=" * 65)
print("HYBRID RETRIEVAL TEST (Dense + BM25 via RRF)")
print("=" * 65)

results_hybrid = retrieve_hybrid(test_query, k=3)

for rank, r in enumerate(results_hybrid, start=1):
    print(f"\nRank {rank}  |  Chunk #{r['chunk_index']}  |  RRF Score: {r['score']}")
    print(r["text"][:200] + "...")
print("=" * 65)


## Step 10 — Compare All Three Retrievers Side by Side

Run the same question through Dense, BM25, and Hybrid.
Observe which chunks each method picks and whether they agree.


In [ ]:
def compare_retrievers(query, k=3):
    """
    Run all three retrieval methods on the same query and print a comparison.

    Args:
        query (str): The question to test
        k (int): Number of results per method
    """
    print("=" * 70)
    print(f"QUERY: {query}")
    print("=" * 70)

    methods = {
        "Dense (MPNet)"  : retrieve_dense(query,  k=k),
        "BM25 (Keyword)" : retrieve_bm25(query,   k=k),
        "Hybrid (RRF)"   : retrieve_hybrid(query, k=k),
    }

    for method_name, results in methods.items():
        print(f"\n── {method_name} ──")
        for rank, r in enumerate(results, start=1):
            # Show: rank, chunk index, score, first 120 chars
            print(f"  [{rank}] Chunk #{r['chunk_index']:>3}  Score: {r['score']:.4f}  "
                  f"| {r['text'][:120].strip()}...")

    print("\n" + "=" * 70)


# ── Run comparison on a sample query ──
compare_retrievers(
    query = "What is the main topic of this document?",
    k     = 3
)


## Step 11 — Generate a Grounded Answer with Claude Haiku

The retrieved chunks are assembled into a context block and injected into
a **grounding system prompt** that instructs Claude Haiku to:

1. Answer **only** from the provided context
2. Say "I don't have this information in the PDF" if the answer isn't there
3. Cite the chunk number that supports its answer

This prevents hallucination — Claude cannot invent facts outside the PDF.


In [ ]:
import anthropic  # Official Anthropic SDK

# Initialise the Anthropic client (reads ANTHROPIC_API_KEY from environment)
CLAUDE_CLIENT = anthropic.Anthropic()

# ── Grounding system prompt ──
# {context} is a placeholder filled in at query time
GROUNDING_SYSTEM_PROMPT = """You are a document Q&A assistant.

Rules you must follow:
1. Answer ONLY using the information in the context provided below.
2. Do NOT use your own training knowledge — only the context counts.
3. If the answer is not present in the context, respond with exactly:
   "I don't have this information in the provided PDF."
4. Cite the chunk number (e.g. [Chunk 4]) that supports each part of your answer.
5. Be concise and directly answer the question.

Context:
{context}"""


def build_context_block(retrieved_chunks):
    """
    Format a list of retrieved chunks into a numbered context string.

    Args:
        retrieved_chunks (list[dict]): Output of any retrieve_* function

    Returns:
        str: Formatted context block for the system prompt
    """
    parts = []
    for r in retrieved_chunks:
        # Label each chunk clearly so Claude can cite it
        label = f"[Chunk {r['chunk_index']}]"
        parts.append(label + "\n" + r["text"])

    # Join chunks with a visible separator
    return ("\n" + "─" * 50 + "\n").join(parts)


def ask_claude(user_question, retrieval_method="hybrid", k=3):
    """
    Full RAG pipeline: retrieve → build context → generate grounded answer.

    Args:
        user_question (str): The question to answer
        retrieval_method (str): 'dense', 'bm25', or 'hybrid'
        k (int): Number of chunks to retrieve

    Returns:
        dict: {answer, retrieved_chunks, method_used}
    """
    # ── Retrieve ──
    if retrieval_method == "dense":
        retrieved = retrieve_dense(user_question, k=k)
    elif retrieval_method == "bm25":
        retrieved = retrieve_bm25(user_question, k=k)
    else:  # default: hybrid
        retrieved = retrieve_hybrid(user_question, k=k)

    # ── Build context ──
    context_block = build_context_block(retrieved)  # Numbered chunk text

    # ── Fill in system prompt ──
    system_prompt = GROUNDING_SYSTEM_PROMPT.format(context=context_block)

    # ── Call Claude Haiku ──
    response = CLAUDE_CLIENT.messages.create(
        model      = "claude-haiku-4-5-20251001",    # Claude Haiku — fast and cost-efficient
        max_tokens = 1024,                  # Cap response length
        system     = system_prompt,         # Grounding instructions + context
        messages   = [
            {"role": "user", "content": user_question}  # User's actual question
        ]
    )

    answer = response.content[0].text  # Extract the text from Claude's response

    return {
        "answer"           : answer,
        "retrieved_chunks" : retrieved,
        "method_used"      : retrieval_method
    }


print("✅ Claude Haiku pipeline ready")
print("   Use: ask_claude('your question here')")


## Step 12 — Ask Your First Question

Type any question about the PDF content. The pipeline will:
- Retrieve the 3 most relevant chunks (Hybrid by default)
- Pass them to Claude Haiku with the grounding prompt
- Return the answer with chunk citations


In [ ]:
# ── Ask a question using Hybrid retrieval (recommended default) ──
YOUR_QUESTION = "What is this document about?"  # <── Change this to your own question

result = ask_claude(
    user_question    = YOUR_QUESTION,
    retrieval_method = "hybrid",  # Try 'dense' or 'bm25' or 'hybrid'
    k                = 3          # Number of chunks to retrieve
)

print("=" * 65)
print(f"QUESTION: {YOUR_QUESTION}")
print("=" * 65)
print(f"\nANSWER ({result['method_used']} retrieval):")
print(result["answer"])

print("\n── Chunks retrieved ──")
for r in result["retrieved_chunks"]:
    print(f"  Chunk #{r['chunk_index']}  Score: {r['score']:.4f}  | "
          f"{r['text'][:100].strip()}...")


## Step 13 — Compare All Three Methods on the Same Question

This cell runs your question through all three retrieval strategies
and shows each answer side by side so you can judge the difference.


In [ ]:
def compare_all_methods(question, k=3):
    """
    Ask the same question using Dense, BM25, and Hybrid retrieval.
    Print all three answers for comparison.

    Args:
        question (str): The question to ask
        k (int): Chunks to retrieve per method
    """
    print("=" * 65)
    print(f"QUESTION: {question}")
    print("=" * 65)

    for method in ["dense", "bm25", "hybrid"]:
        result = ask_claude(question, retrieval_method=method, k=k)

        print(f"\n── Method: {method.upper()} ──")
        print(result["answer"])

        # Show which chunks each method selected
        chunk_ids = [f"#{r['chunk_index']}" for r in result["retrieved_chunks"]]
        print(f"   Retrieved chunks: {', '.join(chunk_ids)}")

    print("\n" + "=" * 65)


# ── Run comparison ──
YOUR_COMPARISON_QUESTION = "What is this document about?"  # <── Change me

compare_all_methods(YOUR_COMPARISON_QUESTION, k=3)


## Step 14 — Free-Form Q&A Session

Run this cell to enter an interactive Q&A loop.
Type any question and press Enter. Type `quit` to stop.
You can choose the retrieval method at the start.


In [ ]:
# Interactive Q&A loop
# Run the cell, then type questions in the text box that appears below

print("PDF Q&A Session")
print("Retrieval methods: dense | bm25 | hybrid")
print("Type 'quit' to exit")
print("=" * 50)

# Let the user pick retrieval method once for the session
chosen_method = input("Choose retrieval method [hybrid]: ").strip().lower()
if chosen_method not in ("dense", "bm25", "hybrid"):
    chosen_method = "hybrid"  # Default to hybrid if invalid input
print(f"Using: {chosen_method} retrieval\n")

while True:
    question = input("Your question: ").strip()  # Read question from user

    if question.lower() == "quit":              # Exit condition
        print("Session ended.")
        break

    if not question:                            # Skip empty input
        continue

    result = ask_claude(question, retrieval_method=chosen_method, k=3)

    print(f"\nAnswer: {result['answer']}")

    # Show which chunks were used
    chunk_ids = [f"#{r['chunk_index']}" for r in result["retrieved_chunks"]]
    print(f"(Retrieved chunks: {', '.join(chunk_ids)})\n")
    print("-" * 50)


## Step 15 — Test Grounding: Does Claude Know When to Say "I Don't Know"?

A well-grounded system should decline questions that cannot be answered
from the PDF rather than inventing an answer.

This cell asks five questions that are clearly outside the PDF's scope.
A good grounding score is ≥ 80% correct deferrals.


In [ ]:
# Questions that should NOT be answerable from the uploaded PDF
out_of_scope = [
    "What is the capital of France?",
    "Who invented the telephone?",
    "What is the boiling point of water?",
    "How do I bake a chocolate cake?",
    "What is the current price of Bitcoin?",
]

# Phrases Claude uses when it correctly declines
deferral_phrases = [
    "don't have this information",
    "not in the provided pdf",
    "not present in",
    "not mentioned",
    "cannot find",
    "no information",
    "not provided",
]

print("GROUNDING TEST — Out-of-Scope Questions")
print("=" * 65)
print("Checking if Claude correctly says 'I don't know' for each question.")
print("-" * 65)

correct = 0  # Count of questions correctly deferred

for q in out_of_scope:
    result = ask_claude(q, retrieval_method="hybrid", k=3)
    answer = result["answer"].lower()  # Lowercase for comparison

    # Check whether any deferral phrase appears in the answer
    correctly_deferred = any(phrase in answer for phrase in deferral_phrases)
    status = "✓" if correctly_deferred else "✗ (possible hallucination)"

    print(f"\n{status}")
    print(f"  Q: {q}")
    print(f"  A: {result['answer'][:120]}...")

    if correctly_deferred:
        correct += 1  # Increment correct count

# Report grounding accuracy
total    = len(out_of_scope)
accuracy = (correct / total) * 100

print("\n" + "=" * 65)
print(f"Grounding accuracy: {accuracy:.0f}%  ({correct}/{total} correctly deferred)")

if accuracy >= 80:
    print("✅ Good — the grounding prompt is working correctly.")
elif accuracy >= 50:
    print("⚠️  Partial — some hallucinations detected. Try strengthening the system prompt.")
else:
    print("❌ Poor — many hallucinations. Review and tighten the grounding instructions.")


---
## Summary — What You Built

| Step | What happened |
|------|--------------|
| 2 | Uploaded a PDF to Colab |
| 3 | Extracted and cleaned text with `pdfplumber` |
| 4 | Split into overlapping 400-token chunks |
| 5 | Embedded each chunk with `all-mpnet-base-v2` (768 dims) |
| 6 | Stored text + vectors + metadata in ChromaDB |
| 7–9 | Implemented **Dense**, **BM25**, and **Hybrid** retrieval |
| 10 | Compared all three retrievers side by side |
| 11 | Built grounded answer generation with **Claude Haiku** |
| 12–14 | Ran interactive Q&A with any question |
| 15 | Verified grounding (Claude says "I don't know" when appropriate) |

### Three Retrieval Methods — Quick Reference

| Method | Best for | Weakness |
|--------|----------|---------|
| **Dense** | Semantic questions, paraphrased queries | Exact keyword misses |
| **BM25** | Specific names, codes, exact terms | No semantic understanding |
| **Hybrid (RRF)** | General use — combines both | Slightly more compute |

**Recommended default:** Hybrid retrieval with chunk size 400 and overlap 50.
